## Actividad 3_17: Regresión con RN. ¿Cuanto vale un portátil?
<div style="border-style:groove;border-width:thin;padding:10px">
Ya hemos hecho un ejercicio de clasificación con redes neuronales. Vamos a trabajar ahora con uno de regresión. En este caso es un dataset con datos de portátiles. 
</div>

<div style="border-style:groove;border-width:thin;padding:10px">
Debes hacer lo siguiente:
    <ol>
        <li>Carga el archivo "laptop_price.csv".</li>
        <li>Echa un vistazo a los datos. Corrige cosas si es necesario.</li>
        <li>Hay muchas columnas categóricas. Transfórmalas.</li>
        <li>Divide el dataset en conjuntos de training y de test.</li>
        <li>Soluciona el ejercicio con una red neuronal.</li>
    </ol>
</div>

In [2611]:
# Datos y preprocesamiento
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Modelos de Clasificación
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# Modelos de Regresión
from sklearn.svm import SVR, LinearSVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from tensorflow import keras

# Métricas de Clasificación
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Métricas de Regresión
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

print("✅ Todas las librerías importadas correctamente")

✅ Todas las librerías importadas correctamente


In [2612]:

df_prices = pd.read_csv('laptop_price.csv', encoding='latin-1')

df_prices.head()

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_euros
0,1,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37kg,1339.69
1,2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
2,3,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,1.86kg,575.00
3,4,Apple,MacBook Pro,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16GB,512GB SSD,AMD Radeon Pro 455,macOS,1.83kg,2537.45
4,5,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8GB,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37kg,1803.60


In [2613]:
df_prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1303 entries, 0 to 1302
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   laptop_ID         1303 non-null   int64  
 1   Company           1303 non-null   object 
 2   Product           1303 non-null   object 
 3   TypeName          1303 non-null   object 
 4   Inches            1303 non-null   float64
 5   ScreenResolution  1303 non-null   object 
 6   Cpu               1303 non-null   object 
 7   Ram               1303 non-null   object 
 8   Memory            1303 non-null   object 
 9   Gpu               1303 non-null   object 
 10  OpSys             1303 non-null   object 
 11  Weight            1303 non-null   object 
 12  Price_euros       1303 non-null   float64
dtypes: float64(2), int64(1), object(10)
memory usage: 132.5+ KB


In [2614]:
df_prices['Weight'].unique()

array(['1.37kg', '1.34kg', '1.86kg', '1.83kg', '2.1kg', '2.04kg', '1.3kg',
       '1.6kg', '2.2kg', '0.92kg', '1.22kg', '0.98kg', '2.5kg', '1.62kg',
       '1.91kg', '2.3kg', '1.35kg', '1.88kg', '1.89kg', '1.65kg',
       '2.71kg', '1.2kg', '1.44kg', '2.8kg', '2kg', '2.65kg', '2.77kg',
       '3.2kg', '0.69kg', '1.49kg', '2.4kg', '2.13kg', '2.43kg', '1.7kg',
       '1.4kg', '1.8kg', '1.9kg', '3kg', '1.252kg', '2.7kg', '2.02kg',
       '1.63kg', '1.96kg', '1.21kg', '2.45kg', '1.25kg', '1.5kg',
       '2.62kg', '1.38kg', '1.58kg', '1.85kg', '1.23kg', '1.26kg',
       '2.16kg', '2.36kg', '2.05kg', '1.32kg', '1.75kg', '0.97kg',
       '2.9kg', '2.56kg', '1.48kg', '1.74kg', '1.1kg', '1.56kg', '2.03kg',
       '1.05kg', '4.4kg', '1.90kg', '1.29kg', '2.0kg', '1.95kg', '2.06kg',
       '1.12kg', '1.42kg', '3.49kg', '3.35kg', '2.23kg', '4.42kg',
       '2.69kg', '2.37kg', '4.7kg', '3.6kg', '2.08kg', '4.3kg', '1.68kg',
       '1.41kg', '4.14kg', '2.18kg', '2.24kg', '2.67kg', '2.14kg',
       '1.

In [2615]:
df_prices.drop(columns=['laptop_ID', 'Product'], inplace=True)

In [2616]:
for index, value in df_prices['Weight'].items():
    df_prices.at[index, 'Weight'] = float(value.replace('kg', '').strip())

In [2617]:
for index, value in df_prices['Ram'].items():
    df_prices.at[index, 'Ram'] = int(value.replace('GB', '').strip())

In [2618]:
df_prices['Ram'] = df_prices['Ram'].astype(int)
df_prices['Weight'] = df_prices['Weight'].astype(float)

In [2619]:
df_prices.corr(numeric_only=True)['Price_euros'].abs().sort_values(ascending=False)[1:]

Ram       0.743007
Weight    0.210370
Inches    0.068197
Name: Price_euros, dtype: float64

In [2620]:
df_prices = pd.get_dummies(df_prices, dtype=int)

In [2621]:
df_prices.head()

,Inches,Ram,Weight,Price_euros,Company_Acer,Company_Apple,Company_Asus,Company_Chuwi,Company_Dell,Company_Fujitsu,...,Gpu_Nvidia Quadro M620M,OpSys_Android,OpSys_Chrome OS,OpSys_Linux,OpSys_Mac OS X,OpSys_No OS,OpSys_Windows 10,OpSys_Windows 10 S,OpSys_Windows 7,OpSys_macOS
0,13.3,8,1.37,1339.69,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,13.3,8,1.34,898.94,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,15.6,8,1.86,575.00,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,15.4,16,1.83,2537.45,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,13.3,8,1.37,1803.60,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [2622]:
X = df_prices.drop(columns=['Price_euros'])
y = df_prices['Price_euros']

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [2623]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
X, y, test_size=0.1, random_state=0)

X_train, X_valid, y_train, y_valid = train_test_split(
X_train_full, y_train_full, test_size=0.1, random_state=0)

In [2624]:
model = keras.models.Sequential([
keras.layers.Dense(210, activation="relu", input_shape=X_train.shape[1:]),
keras.layers.Dense(140, activation="relu"),
keras.layers.Dense(1)
])

model.compile(loss="mean_absolute_error", optimizer="sgd",metrics=['mae'])


In [2625]:
history = model.fit(X_train, y_train, epochs=300,
validation_data=(X_valid, y_valid))

mse_test = model.evaluate(X_test, y_test)

Epoch 1/300
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1089.3641 - mae: 1089.3641 - val_loss: 1079.7389 - val_mae: 1079.7389
Epoch 2/300
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 613.7023 - mae: 613.7023 - val_loss: 337.8569 - val_mae: 337.8569
Epoch 3/300
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 267.6776 - mae: 267.6776 - val_loss: 262.7666 - val_mae: 262.7666
Epoch 4/300
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 215.8203 - mae: 215.8203 - val_loss: 264.5510 - val_mae: 264.5510
Epoch 5/300
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 195.3153 - mae: 195.3153 - val_loss: 253.1683 - val_mae: 253.1683
Epoch 6/300
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 188.2505 - mae: 188.2505 - val_loss: 290.8028 - val_mae: 290.8028
Epoch 7/300
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 180.6861 - mae: 180.6861 - val_loss: 231.5653 - val_mae: 231.5653
Epoch 8/300
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 191.2650 - mae: 191.2650 - val_loss: 238.2807 - val_mae: 238.2

In [2626]:
X_new = X_test[:3] 
y_pred = model.predict(X_new)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step


In [2627]:
y_pred = model.predict(X_test)
r2_score(y_test, y_pred)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 


0.8182715813953968

In [2629]:
model.save("price_predictor.keras")